In [ ]:
# Paso 1: Cargar dataset y verificar estructura
import pandas as pd

# Cargar archivo CSV (ajusta el nombre si es diferente)
df = pd.read_csv('student_mental_health_burnout_1M.csv')

# Mostrar primeras filas y nombres de columnas
print("Primeras 5 filas:")
print(df.head())
print("\nNombres de columnas:")
print(df.columns.tolist())
print(f"\nDimensiones: {df.shape[0]} filas, {df.shape[1]} columnas")
print("\nTipos de datos:")
print(df.dtypes)

Primeras 5 filas:
   age gender  academic_year  study_hours_per_day  exam_pressure  \
0   23   Male              2             5.596071       6.487218   
1   20   Male              3             5.597171       5.631481   
2   29   Male              2             2.580491       6.015297   
3   27   Male              4             4.607208       6.684005   
4   24   Male              4             2.186569       4.010945   

   academic_performance  stress_level  anxiety_score  depression_score  \
0             68.411114      4.116950       2.275713          1.986730   
1             67.682159      0.349489       0.000000          0.000000   
2             58.372363      3.476177       2.425201          0.851996   
3             68.925653      6.778843       4.512425          4.285645   
4             69.141915      1.854595       1.102558          0.000000   

   sleep_hours  physical_activity  social_support  screen_time  \
0     6.880545           2.728861        6.470080     4.993801

In [ ]:
# Paso 2: Preprocesamiento y división train/test
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pandas as pd

# Copias para no modificar el original
X = df.drop(columns=['risk_level']).copy()
y = df['risk_level'].copy()

# 1. Codificar 'gender'
le_gender = LabelEncoder()
X['gender'] = le_gender.fit_transform(X['gender'])   # Male, Female, Other -> 0,1,2

# 2. Identificar columnas numéricas (todas excepto 'gender' que ya es numérica)
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
print("Columnas numéricas a escalar:", numeric_cols)

# 3. Escalar las numéricas
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

# 4. Codificar la variable objetivo risk_level
le_risk = LabelEncoder()
y_encoded = le_risk.fit_transform(y)   # Low->0, Medium->1, High->2

# 5. División entrenamiento (80%) y prueba (20%) con estratificación
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Tamaño entrenamiento: {X_train.shape}")
print(f"Tamaño prueba: {X_test.shape}")
print("Distribución de clases en entrenamiento:")
print(pd.Series(y_train).value_counts().sort_index())

Columnas numéricas a escalar: ['age', 'gender', 'academic_year', 'study_hours_per_day', 'exam_pressure', 'academic_performance', 'stress_level', 'anxiety_score', 'depression_score', 'sleep_hours', 'physical_activity', 'social_support', 'screen_time', 'internet_usage', 'financial_stress', 'family_expectation', 'burnout_score', 'mental_health_index', 'dropout_risk']
Tamaño entrenamiento: (800000, 19)
Tamaño prueba: (200000, 19)
Distribución de clases en entrenamiento:
0     12064
1    613316
2    174620
Name: count, dtype: int64


In [ ]:
# Paso 3: Entrenar MLPClassifier y KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import time

# --- MLPClassifier (Red Neuronal) ---
print("=== Entrenando MLPClassifier ===")
start = time.time()
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    random_state=42
)
mlp.fit(X_train, y_train)
mlp_time = time.time() - start
print(f"Tiempo de entrenamiento MLP: {mlp_time:.2f} segundos")

# Predicción y evaluación
y_pred_mlp = mlp.predict(X_test)
acc_mlp = accuracy_score(y_test, y_pred_mlp)
f1_mlp = f1_score(y_test, y_pred_mlp, average='weighted')
print(f"MLP - Accuracy: {acc_mlp:.4f}, F1-score (weighted): {f1_mlp:.4f}")
print("Reporte de clasificación MLP:")
print(classification_report(y_test, y_pred_mlp, target_names=le_risk.classes_))

# --- KNeighborsClassifier ---
print("\n=== Entrenando KNeighborsClassifier ===")
start = time.time()
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)  # n_jobs=-1 usa todos los núcleos
knn.fit(X_train, y_train)
knn_time = time.time() - start
print(f"Tiempo de entrenamiento KNN: {knn_time:.2f} segundos")

y_pred_knn = knn.predict(X_test)
acc_knn = accuracy_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn, average='weighted')
print(f"KNN - Accuracy: {acc_knn:.4f}, F1-score (weighted): {f1_knn:.4f}")
print("Reporte de clasificación KNN:")
print(classification_report(y_test, y_pred_knn, target_names=le_risk.classes_))

=== Entrenando MLPClassifier ===
Tiempo de entrenamiento MLP: 139.88 segundos
MLP - Accuracy: 0.9977, F1-score (weighted): 0.9977
Reporte de clasificación MLP:
              precision    recall  f1-score   support

        High       1.00      0.94      0.97      3016
         Low       1.00      1.00      1.00    153329
      Medium       0.99      1.00      0.99     43655

    accuracy                           1.00    200000
   macro avg       1.00      0.98      0.99    200000
weighted avg       1.00      1.00      1.00    200000


=== Entrenando KNeighborsClassifier ===
Tiempo de entrenamiento KNN: 0.14 segundos
KNN - Accuracy: 0.9272, F1-score (weighted): 0.9255
Reporte de clasificación KNN:
              precision    recall  f1-score   support

        High       0.86      0.51      0.64      3016
         Low       0.95      0.97      0.96    153329
      Medium       0.85      0.81      0.83     43655

    accuracy                           0.93    200000
   macro avg       0.

In [ ]:
# Paso 4: Guardar modelos y transformadores
import joblib

# Guardar ambos modelos
joblib.dump(mlp, 'mlp_model.pkl')
joblib.dump(knn, 'knn_model.pkl')

# Guardar transformadores
joblib.dump(le_gender, 'le_gender.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le_risk, 'le_risk.pkl')

# Guardar lista de columnas numéricas (para escalado)
joblib.dump(numeric_cols, 'numeric_cols.pkl')

# Guardar el orden de las columnas que usó el modelo (importante para la API)
feature_order = X.columns.tolist()  # X ya tiene las columnas en el orden final
joblib.dump(feature_order, 'feature_order.pkl')

print("Artefactos guardados correctamente:")
print(" - mlp_model.pkl")
print(" - knn_model.pkl")
print(" - le_gender.pkl")
print(" - scaler.pkl")
print(" - le_risk.pkl")
print(" - numeric_cols.pkl")
print(" - feature_order.pkl")

Artefactos guardados correctamente:
 - mlp_model.pkl
 - knn_model.pkl
 - le_gender.pkl
 - scaler.pkl
 - le_risk.pkl
 - numeric_cols.pkl
 - feature_order.pkl
